In [ ]:
# Stage 1: Generate 10 test cases
import anthropic
import json
import re

client = anthropic.Anthropic()

# The system role is the most reliable way to enforce 'JSON ONLY' output
system_instruction = "You are a data generator. You output ONLY a raw JSON array. No markdown, no conversation, no preamble."

prompt = """Generate 10 realistic Rutgers University New Brunswick student profiles for testing a scheduling AI.
Return a JSON array where each object has these fields: id, semester, subjects, core_elective, credits_wanted, credits_completed, academic_history.

Rules for format:
- 'subjects': Must be a string of comma-separated major/minor codes (e.g., "198, 640").
- 'academic_history': Must be a comma-separated string of course codes (e.g., "01:198:111, 01:640:151").
- 'core_elective': Must be a string (e.g., "CCD" or empty).

Content Rules:
- Vary semester (Freshman-Senior), credits (12-18), and core_electives (CCO, CCD, HST, SCL, NS, AH, WC or empty).
- Include 2 edge cases: one difficult-to-fill credit request, one Freshman Fall with no history.
- Ensure all 10 profiles have unique or varied major/minor subject combinations.
"""

response = client.messages.create(
    model="claude-3-5-sonnet-20240620",
    system=system_instruction,
    max_tokens=2500,
    messages=[{"role": "user", "content": prompt}]
)

raw_text = response.content[0].text
print("--- Raw Response Start ---")
print(raw_text)
print("--- Raw Response End ---")

# Regex to find the JSON array regardless of surrounding text
match = re.search(r'\[.*\]', raw_text, re.DOTALL)

if match:
    try:
        test_cases = json.loads(match.group(0))
        with open("test_cases.json", "w") as f:
            json.dump(test_cases, f, indent=2)
        print(f"\nSuccess! Saved {len(test_cases)} test cases to test_cases.json")
    except json.JSONDecodeError as e:
        print(f"\nFailed to parse JSON: {e}")
else:
    print("\nError: Could not find a JSON array [...] in the AI response.")

In [17]:
# Stage 2: Evaluate test cases
import asyncio
import aiohttp
import json
import os
from anthropic import Anthropic

client = Anthropic()

# 1. Fetch and Filter Logic
async def fetch_and_filter_rutgers_courses(subjects_string: str, core_string: str, history_string: str):
    url = "https://classes.rutgers.edu/soc/api/courses.json?year=2026&term=9&campus=NB&level=U"
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(url) as resp:
                if resp.status != 200:
                    return f"Error: API returned status {resp.status}"
                all_courses = await resp.json()
    except Exception as e:
        return f"Error fetching API: {e}"

    # Ensure this matches your bot's exact mapping
    code_map = {
        "CCO": "CC-O", "CCD": "CC-D", "NS": "NS", "SCL": "SCL", 
        "HST": "HST", "AH": "AH", "WC": "WC", "WCr": "WCr"
    }
    
    target_subjects = [s.strip() for s in subjects_string.split(",")] if subjects_string else []
    user_input = core_string.strip().upper()
    target_core = code_map.get(user_input, user_input) 
    history_check = history_string if history_string else ""

    filtered_courses = []
    core_elective_count = 0

    for course in all_courses:
        course_str = course.get("courseString", "")
        # Skip if already in academic history
        if course_str and course_str in history_check:
            continue
            
        subj = str(course.get("subject", ""))
        is_major_minor = subj in target_subjects
        
        # Core check
        core_codes = [c.get("code") for c in course.get("coreCodes", [])]
        has_core = target_core in core_codes if target_core else False
        
        if is_major_minor:
            filtered_courses.append({"code": course_str, "title": course.get("title"), "credits": course.get("credits"), "prereqs": course.get("preReqNotes", ""), "coreCodes": core_codes})
        elif has_core and core_elective_count < 10:
            filtered_courses.append({"code": course_str, "title": course.get("title"), "credits": course.get("credits"), "prereqs": course.get("preReqNotes", ""), "coreCodes": core_codes})
            core_elective_count += 1
            
    return json.dumps(filtered_courses)

# 2. Chat Function (System Prompt)
def chat(user_prompt):
    system = """
    You are RU Scarlet Scheduler AI, a course scheduling assistant for Rutgers University New Brunswick students.

    <examples>
        <example>
            <user_input>
                Plan my semester: Freshman Spring. Major: 960, 198. Core: NS. Credits: 14. Completed: 01:198:111, 01:960:151.
            </user_input>
            <ideal_response>
                - 01:198:112 DATA STRUCTURES | 4 Credits | Prerequisites met: Yes
                - 01:198:205 INTR DISCRET STRCT I | 4 Credits | Prerequisites met: Yes
                - 01:119:103 PRINCIPLES OF BIOL | 3 Credits | Prerequisites met: Yes (NS)
                - 01:198:104 GREAT IDEAS IN CS | 1 Credit | Prerequisites met: Yes

                This schedule covers core CS and Statistics major requirements while fulfilling the NS Core Elective. The workload is appropriate for a Freshman Spring student, though it lands at 12 credits as no eligible course could fill the remaining gap without unmet prerequisites.
            </ideal_response>
        </example>
        <example>
            <user_input>
                Plan my semester: Senior Fall. Major: 260, 920, 640. Core: CCO. Credits: 11. Completed: 01:260:201, 01:260:251, 01:920:201, 01:920:301, 01:640:150, 01:640:251, 01:198:111, 01:198:251.
            </user_input>
            <ideal_response>
                - 01:920:311 SOC RESEARCH | 4 Credits | Prerequisites met: Yes
                - 01:920:316 SOC THEORY | 4 Credits | Prerequisites met: Partial (College Writing assumed met as a Senior)
                - 01:920:225 INTRO IMMIGRATION | 3 Credits | Prerequisites met: Yes (CCO)

                This schedule meets the 11-credit goal exactly, covering core Sociology requirements and satisfying the CCO Core Elective. The workload is very manageable for a Senior Fall semester.
            </ideal_response>
        </example>
        <example>
            <user_input>
                Plan my semester: Junior Fall. Major: 440, 960, 198. Core: HST. Credits: 16. Completed: 01:440:101, 01:440:201, 01:960:151, 01:960:251, 01:198:111, 01:198:112.
            </user_input>
            <ideal_response>
                - 01:198:205 INTR DISCRET STRCT I | 4 Credits | Prerequisites met: Yes
                - 01:198:211 COMPUTER ARCHITECTUR | 4 Credits | Prerequisites met: Yes
                - 01:198:213 SOFTWARE METHODOLOGY | 4 Credits | Prerequisites met: Yes
                - 01:016:222 MODERN AFRICA | 3 Credits | Prerequisites met: Yes (HST)

                This schedule covers three core CS requirements and satisfies the HST Core Elective, aligning well with junior-level progression. It lands at 15 credits as no eligible course could fill the remaining 1-credit gap without exceeding the target.
            </ideal_response>
        </example>
    </examples>

    <rules>
    - Only recommend real Rutgers courses found in the LIVE API DATA provided.
    - Never recommend a course whose prerequisites the student has not yet met. Skip it silently.
    - Prioritize courses in this exact order: Major/Minor → Requested Core Elective → Free Electives.
    - Only suggest a Core Elective if explicitly requested by the student.
    - Never suggest free electives unless Major/Minor and Core courses cannot fill the requested credits.
    - DO NOT relist or mention courses the student has already taken. Ever.
    - DO NOT offer alternative schedules or multiple options. Provide exactly one recommended schedule.
    - DO NOT show your reasoning process, thinking, scratchpad, or eligibility analysis. Output the final schedule only.
    - Try to meet the exact credit count requested. If not possible, go under rather than over.
    - If you cannot reach the requested credit count, mention the shortfall in the summary only.
    - Your summary must be exactly 2-3 sentences. If it exceeds 3 sentences, delete sentences until it does.
    - Never offer alternative schedules or swaps in the summary.
    </rules>

    <output_format>
    For each recommended course:
    - Course code and name | Credits | Prerequisites met: Yes / No / Partial

    End with a summary of exactly 2-3 sentences that covers:
    1. Whether the schedule meets the student's major requirements and credit goal
    2. Whether the overall workload is reasonable for their level
    3. A brief credit warning only if you fell short, or if total is below 12 or above 18
    Do not explain individual course choices in the summary.
    </output_format>
    """
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[{"role": "user", "content": user_prompt}],
        system=system,
        temperature=0.1,
    )
    return message.content[0].text

# 3. Execution Wrapper
async def run_all():
    with open("test_cases.json") as f:
        test_cases = json.load(f)

    results = []
    for case in test_cases[:5]:
        print(f"Running test case {case['id']}...")
        subj_input = ", ".join(case["subjects"]) if isinstance(case["subjects"], list) else case["subjects"]
        
        api_data = await fetch_and_filter_rutgers_courses(
            subj_input, case.get("core_elective", ""), case.get("academic_history", "")
        )

        user_prompt = f"Plan my semester: {case['semester']}. Major: {subj_input}. Core: {case.get('core_elective')}. Credits: {case['credits_wanted']}. Completed: {case['academic_history']}. API: {api_data}"

        response = chat(user_prompt)
        results.append({"id": case["id"], "input": case, "output": response})

    with open("test_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Done! Results saved to test_results.json")

await run_all()

Running test case RU001...
Running test case RU002...
Running test case RU003...
Running test case RU004...
Running test case RU005...
Done! Results saved to test_results.json


In [18]:
# Stage 3: Grading Output

import json
import re
from anthropic import Anthropic

client = Anthropic()

system_prompt = """
You are an expert AI evaluator.

Grade the RU Scarlet Scheduler AI using the rubric provided.

Be strict about:
- Incorrect prerequisites
- Fake courses
- Incorrect credit counts
- Ignored constraints

Return ONLY valid JSON.

Keep the summary under 2 sentences (max 50 words).
Focus only on the biggest strengths and weaknesses.
"""

rubric = """
1. Accuracy (0-3 pts):
   - Assume all recommended courses are real and valid IF they follow the correct Rutgers course code format (XX:XXX:XXX). Do NOT penalize for unrecognized course names or titles.
   - Only deduct points if: a course code does NOT follow the XX:XXX:XXX format, OR if a prerequisite is clearly ignored based on the student's provided course history.

2. Constraint Adherence (0-3 pts):
   - Did it stay at or under the requested credit count?
   - Do NOT penalize for undershooting credits if the AI acknowledged the shortfall in the summary.
   - Did it provide exactly one schedule with no alternatives offered?
   - Did it satisfy Major/Minor and Core requirements requested?

3. Completeness & Conciseness (0-2 pts):
   - Did the output finish without cutoff?
   - Did it avoid relisting courses the student already took?
   - Was reasoning saved for the summary only with no inline explanations?

4. Professionalism & Tone (0-2 pts):
   - Is the summary 2-3 sentences and helpful?
   - Only deduct if the tone is genuinely unhelpful, confusing, or unprofessional.
   - Do NOT deduct purely for being slightly over a word count. Focus on whether it reads well and is actionable for a student.
"""

def extract_text(response):
    text_parts = []
    for block in response.content:
        if getattr(block, "type", None) == "text":
            text_parts.append(block.text)
    return "\n".join(text_parts)

def calculate_total(parsed):
    b = parsed.get("breakdown", {})
    return (
        b.get("Accuracy", 0) +
        b.get("Constraints", 0) +
        b.get("Completeness", 0) +
        b.get("Professionalism", 0)
    )

def grade(result):
    user_prompt = f"""
Please grade the following AI response.

RUBRIC:
{rubric}

STUDENT INPUT:
{json.dumps(result["input"], indent=2)}

AI OUTPUT:
{result["output"]}

Return ONLY valid JSON in this format:

{{
  "total_score": 0,
  "breakdown": {{
    "Accuracy": 0,
    "Constraints": 0,
    "Completeness": 0,
    "Professionalism": 0
  }},
  "summary": "Short explanation"
}}
"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}]
    )

    raw_text = extract_text(response)

    try:
        parsed = json.loads(raw_text)
        parsed["id"] = result["id"]
        parsed["total_score"] = calculate_total(parsed)
        return parsed

    except Exception:
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                parsed["id"] = result["id"]
                parsed["total_score"] = calculate_total(parsed)
                return parsed
            except Exception:
                pass

    return {
        "id": result["id"],
        "total_score": 0,
        "breakdown": {
            "Accuracy": 0,
            "Constraints": 0,
            "Completeness": 0,
            "Professionalism": 0
        },
        "summary": "Failed to parse Claude output"
    }

# Execution
with open("test_results.json") as f:
    results = json.load(f)

grades = []

for result in results[:5]:
    print(f"Grading test case {result['id']}...")
    grades.append(grade(result))

with open("grades.json", "w") as f:
    json.dump(grades, f, indent=2)

print("Grading complete!")

Grading test case RU001...
Grading test case RU002...
Grading test case RU003...
Grading test case RU004...
Grading test case RU005...
Grading complete!


In [4]:
# Stage 4: Export Evaluation Dashboard
import json
import webbrowser
import os

# Load Data
with open("grades.json") as f:
    grades = json.load(f)
with open("test_results.json") as f:
    results = json.load(f)

results_map = {r["id"]: r for r in results}

def get_total_bg_color(score):
    if score >= 10: return "#2e8f33"
    if score >= 9: return "#1e4620"
    if score >= 8: return "#8BA543"
    if score >= 7: return "#665400"
    if score >= 4: return "#85451a"
    return "#721c24"

# Metrics
total_tests = len(grades)
avg_score = round(sum(g["total_score"] for g in grades) / total_tests, 2) if total_tests > 0 else 0
passed_tests = len([g for g in grades if g["total_score"] >= 7])
pass_rate = round((passed_tests / total_tests) * 100, 1) if total_tests > 0 else 0
best_case = max(grades, key=lambda x: x["total_score"]) if total_tests > 0 else {"id": "N/A", "total_score": 0}
worst_case = min(grades, key=lambda x: x["total_score"]) if total_tests > 0 else {"id": "N/A", "total_score": 0}

import json
import webbrowser
import os

rows = ""
for g in grades:
    r = results_map.get(g["id"], {})
    inp = r.get("input", {})
    b = g.get("breakdown", {})
    if not b: b = g
    
    ai_output = r.get("output", "No output generated.")
    # Calculate score background
    score = g.get("total_score", 0)
    tot_bg = get_total_bg_color(score)

    rows += f"""
    <tr id="case-{g['id']}">
        <td>{g.get("id", "N/A")}</td>
        <td style="font-size: 1em; line-height: 1.4;">
            <strong>Semester:</strong> {inp.get('semester', 'N/A')}<br>
            <strong>Major:</strong> {inp.get('subjects', 'N/A')}<br>
            <strong>Elective Core:</strong> {inp.get('core_elective', 'N/A')}<br>
            <strong>Credits Wanted:</strong> {inp.get('credits_wanted', 'N/A')}<br>
            <strong>Credits Completed:</strong> {inp.get('credits_completed', 'N/A')}<br>
            <strong>Course History:</strong> {inp.get('academic_history', 'N/A')}
        </td>
        <td style="font-size: 1em; line-height: 1.6; white-space: nowrap;">
            Accuracy: {b.get('Accuracy', 0)}/3<br>
            Constraints: {b.get('Constraints', 0)}/3<br>
            Completeness: {b.get('Completeness', 0)}/2<br>
            Professionalism: {b.get('Professionalism', 0)}/2
        </td>
        <td>
            <div class="scrollable-output">{ai_output}</div>
        </td>
        <td style="text-align: center; vertical-align: top;">
            <div style="
                background-color: {tot_bg}; 
                color: #fff; 
                width: 35px; 
                height: 35px; 
                line-height: 35px; 
                border-radius: 6px; 
                display: inline-block; 
                font-weight: bold;
                text-align: center;
            ">
                {score}
            </div>
        </td>
        <td class="reasoning-col" style="font-size: 1em;">{g.get("summary", "No reasoning provided.")}</td>
    </tr>
    """

html = f"""
<!DOCTYPE html>
<html>
<head>
    <style>
        body {{ font-family: 'Segoe UI', sans-serif; background: #000; color: #fff; padding: 20px; }}
        h1 {{ color: #cc0033; }}
        .cards {{ display: flex; gap: 15px; margin-bottom: 20px; }}
        .card {{ background: #1e1e1e; padding: 15px; border-radius: 8px; border: 1px solid #333; text-decoration: none; color: white; transition: 0.2s; }}
        .card:hover {{ background: #333; }}
        table {{ width: 100%; border-collapse: collapse; background: #121212; table-layout: fixed; }}
        th {{ background: #2a2a2a; color: #ccc; padding: 12px; text-align: left; border-bottom: 2px solid #cc0033; font-size: 1em; }}
        td {{ padding: 15px; border-bottom: 1px solid #333; vertical-align: top; font-size: 1em; }}
        
        /* Updated Widths: Subscores down to 10%, Output up to 45% */
        .subscores-col {{ width: 10%; font-size: 0.9em; }}
        .output-col {{ width: 45%; }}
        .reasoning-col {{ width: 20%; }}
        
        .scrollable-output {{ 
            height: 300px; 
            overflow-y: auto; 
            white-space: pre-wrap; 
            font-family: monospace; 
            background: #0a0a0a; 
            padding: 10px; 
            border: 1px solid #444; 
        }}
        tr:hover {{ background: #1e1e1e; }}
    </style>
</head>
<body>
    <h1>RU Scarlet Scheduler AI — Dashboard</h1>
    <div class="cards">
        <div class="card"><h2>Avg Score</h2><p>{avg_score}/10</p></div>
        <div class="card"><h2>Pass Rate</h2><p>{pass_rate}%</p></div>
        <div class="card"><h2>Total</h2><p>{total_tests}</p></div>
        <a href="#case-{best_case['id']}" class="card"><h2>Best Case 👑</h2><p>{best_case['id']} ({best_case['total_score']}/10)</p></a>
        <a href="#case-{worst_case['id']}" class="card"><h2>Worst Case 😡</h2><p>{worst_case['id']} ({worst_case['total_score']}/10)</p></a>
    </div>
    <table>
        <tr>
            <th style="width: 5%;">ID</th>
            <th style="width: 18%;">User Input</th>
            <th class="subscores-col">Subscores</th>
            <th class="output-col">Output</th>
            <th style="width: 5%;">Total</th>
            <th class="reasoning-col">Reasoning</th>
        </tr>
        {rows}
    </table>
</body>
</html>
"""

with open("eval_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)
webbrowser.open("file://" + os.path.abspath("eval_dashboard.html"))

True